## Fetching the data

In [1]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
load_dotenv()
API_KEY = os.getenv("API_KEY_SB")
FRED_LIST= 'fred_daily_series_list.csv'

def get_daily_series(api_key):
    base_url = "https://api.stlouisfed.org/fred/tags/series"
    params = {
        "api_key": api_key,
        "tag_names": "daily",
        "file_type": "json",
        "limit": 1000,
        "offset": 0
    }

    all_series = []

    while True:
        try:
            response = requests.get(base_url, params=params)
            response.raise_for_status()
            data = response.json()

            if "seriess" in data:
                series_chunk = data["seriess"]
                all_series.extend(series_chunk)

                if len(series_chunk) < params["limit"]:
                    break
                params["offset"] += params["limit"]
            else:
                break
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            break
    return all_series



daily_series = get_daily_series(API_KEY)
df = pd.DataFrame(daily_series)
print(f"\nNumber of FRED series with daily data: {len(df)}\n")
# df = df[['id']]
df.to_csv(FRED_LIST, index=False)



Number of FRED series with daily data: 11472



There are different series with daily data, which is significant. We’ll fetch the actual series one by one for years of data. You can adjust the date range according to your preferences by changing START_DATE and END_DATE. Each series will be saved as a CSV file in the folder data.

In [3]:
from io import StringIO

START_DATE = '1970-01-01'
END_DATE   = '2024-12-12'
DATA_FOLDER = 'data_FRED'

In [ ]:
def fetch_data(series_id):
    request = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    request += f"&cosd={START_DATE}"
    request += f"&coed={END_DATE}"
    try:
        response = requests.get(request)
        response.raise_for_status()
        df = pd.read_csv(StringIO(response.text), parse_dates=True)
        # df.rename(
        #     columns={
        #         df.columns[0]: 'ds',
        #         df.columns[1]: 'value',
        #     },
        #     inplace=True,
        # )
        return series_id, df
    except requests.RequestException as e:
        print(f"Error fetching data for {series_id}: {e}")
        return series_id, None

df = pd.read_csv(FRED_LIST)
os.makedirs(DATA_FOLDER, exist_ok=True)
series_ids = df['id'].tolist()

for series_id in series_ids:
    series_id, data = fetch_data(series_id)
    if data is not None:
        filename = os.path.join(DATA_FOLDER, f"{series_id}.csv")
        data.to_csv(filename, index=False)

Error fetching data for IHLIDXAUTPINDUENGI: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Error fetching data for NASDAQNQUSB10102015LMT: HTTPSConnectionPool(host='fred.stlouisfed.org', port=443): Max retries exceeded with url: /graph/fredgraph.csv?id=NASDAQNQUSB10102015LMT&cosd=1970-01-01&coed=2024-12-12 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000002529087AAE0>, 'Connection to fred.stlouisfed.org timed out. (connect timeout=None)'))


Next, we’ll count the number of CSV files in our data_FRED folder to verify that the API successfully fetched all data series we previously identified.

In [4]:
def count_csv_files(folder_path):
    csv_count = 0
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            csv_count += 1
    return csv_count

num_csv_files = count_csv_files(DATA_FOLDER)
print(f"\nNumber of CSV files in the {DATA_FOLDER} folder: {num_csv_files}\n")


Number of CSV files in the data_FRED folder: 11471



We do a quick exploratory data analysis with one data series.

In [5]:
file_path = os.path.join(DATA_FOLDER, 'AAA10Y.csv')
df = pd.read_csv(file_path)

print("\nLast 10 rows of AAA10Y:")
print("=" * 30)
print(df.tail(10))
print("=" * 30)


Last 10 rows of AAA10Y:
      observation_date  AAA10Y
10934       2024-11-29    0.83
10935       2024-12-02    0.79
10936       2024-12-03    0.79
10937       2024-12-04    0.78
10938       2024-12-05    0.80
10939       2024-12-06    0.81
10940       2024-12-09    0.82
10941       2024-12-10    0.81
10942       2024-12-11    0.84
10943       2024-12-12    0.84


## Preparing the data
1. Determining NYSE trading days
We define a function that returns the dates when the New York Stock Exchange (NYSE) is open.

In [6]:
import pandas_market_calendars as mcal
from typing import Optional

stock_exchange = 'NYSE'
def obtain_market_dates(start_date: str, end_date: str, market : Optional[str] ) -> pd.DataFrame:
    se = mcal.get_calendar(market)
    market_open_dates = se.schedule(
        start_date=start_date,
        end_date=end_date,
    )
    return market_open_dates

market_dates = obtain_market_dates(START_DATE,END_DATE, market=stock_exchange)
print(market_dates.head())
print(market_dates.tail())
print(market_dates.shape)
market_dates.to_csv(f'market_dates_{stock_exchange}.csv')

                         market_open              market_close
1970-01-02 1970-01-02 15:00:00+00:00 1970-01-02 20:00:00+00:00
1970-01-05 1970-01-05 15:00:00+00:00 1970-01-05 20:00:00+00:00
1970-01-06 1970-01-06 15:00:00+00:00 1970-01-06 20:00:00+00:00
1970-01-07 1970-01-07 15:00:00+00:00 1970-01-07 20:00:00+00:00
1970-01-08 1970-01-08 15:00:00+00:00 1970-01-08 20:00:00+00:00
                         market_open              market_close
2024-12-06 2024-12-06 14:30:00+00:00 2024-12-06 21:00:00+00:00
2024-12-09 2024-12-09 14:30:00+00:00 2024-12-09 21:00:00+00:00
2024-12-10 2024-12-10 14:30:00+00:00 2024-12-10 21:00:00+00:00
2024-12-11 2024-12-11 14:30:00+00:00 2024-12-11 21:00:00+00:00
2024-12-12 2024-12-12 14:30:00+00:00 2024-12-12 21:00:00+00:00
(13859, 2)


2- Removing missing values
We mask (temporarily) the empty cells, None values, or entries containing only a `.` in our dataset. We will replace them in the next step.

3- Replacing missing values
We replace any missing values with the previous value in the series, or assign 0 if it’s the first value. For robustness, we set a `MAX_MISSING_DATA` threshold of 50%. Any series with more than 50% missing data is discarded and not used in our model.

In [7]:
from typing import Union, Tuple
import logging
MAX_MISSING_DATA = 0.5

def replace_empty_data(df : pd.DataFrame) -> pd.DataFrame:
    mask = df.isin(["", ".", None])
    rows_to_remove = mask.any(axis=1)
    return df.loc[~rows_to_remove]
def handle_missing_data(
        data: pd.DataFrame,
        market_open_dates : pd.DataFrame,
        data_series : str
) -> Tuple[Union[None,pd.DataFrame], Union[pd.DataFrame, None]]:
    modified_data = data.copy()
    market_open_dates["count"] = 0
    date_counts = data['observation_date'].value_counts()

    market_open_dates["count"] = market_open_dates.index.map(
        date_counts
    ).fillna(0)

    missing_dates = market_open_dates.loc[
        market_open_dates["count"] < 1
    ]

    if not missing_dates.empty:
        max_count = (
            len(market_open_dates)
            * MAX_MISSING_DATA
        )

        if len(missing_dates) > max_count:
            logging.warning(
                f"For the series {data_series} there are "
                f"{len(missing_dates)} data points missing, which is greater than the maximum threshold of "
                f"{MAX_MISSING_DATA * 100}%"
            )
            return pd.DataFrame(), None
        else:
            for date, row in missing_dates.iterrows():
                modified_data = insert_missing_date(
                    modified_data, date, 'observation_date'
                )
    return modified_data, missing_dates


def insert_missing_date(
        data: pd.DataFrame,
        date: str,
        date_column: str ) -> pd.DataFrame:
    date = pd.to_datetime(date)
    if date not in data[date_column].values:
        prev_date = (
            data[data[date_column] < date].iloc[-1]
            if not data[data[date_column] < date].empty
            else data.iloc[0]
        )
        new_row = prev_date.copy()
        new_row[date_column] = date
        data = (
            pd.concat([data, new_row.to_frame().T], ignore_index=True)
            .sort_values(by=date_column)
            .reset_index(drop=True)
        )
    return data

4- Processing individual data series
We apply all our previous steps to each data series individually. When we encounter the S&P 500 series (detected via `if 'SP500.csv in csv_file:`), we save it directly in our `model_data` variable, as this will be the target output for our forecasting model.

In [ ]:
import glob
processed_dataframes = []
market_dates_only = market_dates.index.date

for csv_file in glob.glob(os.path.join(DATA_FOLDER, "*.csv")):
    df = pd.read_csv(csv_file)
    df['observation_date'] = pd.to_datetime(df['observation_date'])
    df_correct_dates = df[df['observation_date'].dt.date.isin(market_dates_only)]
    df_cleaned = replace_empty_data(df_correct_dates)
    processed_df, missing_dates = handle_missing_data(df_cleaned,market_dates,os.path.basename(csv_file).split('.')[0])
    if not processed_df.empty:
        processed_df['observation_date'] = pd.to_datetime(processed_df['observation_date'])
        if not missing_dates.empty:
            for missing_date in missing_dates.index:
                missing_date = pd.to_datetime(missing_date)
                if missing_date in processed_df['observation_date'].values:
                    continue

                previous_day_data = processed_df[processed_df['observation_date'] < missing_date].tail(1)

                if previous_day_data.empty:
                    new_row = pd.DataFrame({'observation_date': [missing_date], 'value': [0]})
                else:
                    new_row = previous_day_data.copy()
                    new_row['observation_date'] = missing_date

                processed_df = pd.concat([processed_df, new_row]).sort_values('observation_date').reset_index(drop=True)

print(f"\nNumber of data series remaining after cleanup: {len(processed_dataframes)}\n")

# Not so important things

5- Dimensionality reduction
Keep in mind that dimensionality reduction is beneficial but optional. While we could train our model using all individual data series, many are likely highly correlated (e.g., stock indices like Dow, S&P 500, and Russell). By reducing dimensions, we can enhance both the robustness and performance of our model. 

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
EXPLAINED_VARIANCE = 0.9
MIN_VARIANCE = 1e-10

combined_df = pd.concat([df.set_index('ds') for df in processed_dataframes], axis=1)
combined_df.columns = [f'value_{i}' for i in range(len(processed_dataframes))]

X = combined_df.values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


pca = PCA(n_components=EXPLAINED_VARIANCE, svd_solver='full')
X_pca = pca.fit_transform(X_scaled)

X_pca = X_pca[:, pca.explained_variance_ > MIN_VARIANCE]


pca_df = pd.DataFrame(
    X_pca,
    columns=[f'PC{i+1}' for i in range(X_pca.shape[1])]
)

print(f"\nOriginal number of features: {combined_df.shape[1]}")
print(f"Number of components after PCA: {pca_df.shape[1]}\n")

model_data = model_data.join(pca_df)

We used principal components analysis (PCA) for dimensionality reduction and used an `EXPLAINED_VARIANCE` of 90%. This means that the algorithm will retain enough principal components to explain 90% of the variance in the original features.

It’s worth noting that while we used PCA for dimensionality reduction, which is an effective, widely-used technique and good enough for our use-case, it primarily considers linear relationships between features. However, we will use a transformer-based model in the next step which is designed to capture non-linear relationships in the data. Consequently, PCA may oversimplify or omit complex, non-linear patterns that the model is designed to detect.

## Training the model
We will forecast the S&P 500 index using our data obtained from FRED. For this purpose, we’ll employ the Temporal Fusion Transformer model (TFT) available in the NeuralForecast library. It’s worth noting that any forecasting model from NeuralForecast that incorporates historical exogenous variables would be suitable for this task.

In [ ]:
from neuralforecast.models import TFT
from neuralforecast import NeuralForecast

TRAIN_SIZE = .90
model_data['unique_id'] = 'SPY'
model_data['price'] = model_data['price'].astype(float)
model_data['y'] = model_data['price'].pct_change()
model_data = model_data.iloc[1:]
hist_exog_list = [col for col in model_data.columns if col.startswith('PC')]

train_size = int(len(model_data) * TRAIN_SIZE)
train_data = model_data[:train_size]
test_data = model_data[train_size:]

model = TFT(
    h=1,
    input_size=24,
    hist_exog_list=hist_exog_list,
    scaler_type='robust',
    max_steps=20
)

nf = NeuralForecast(
    models=[model],
    freq='D'
)

nf.fit(df=model_data)

Next, we generate one-day-ahead predictions for each data point in the test set, forecasting the S&P 500 daily returns. We then convert these return forecasts back to price forecasts by multiplying each predicted return by the previous day’s price.

In [ ]:
y_hat_test_ret = pd.DataFrame()
current_train_data = train_data.copy()

y_hat_ret = nf.predict(current_train_data)
y_hat_test_ret = pd.concat([y_hat_test_ret, y_hat_ret.iloc[[-1]]])

for i in range(len(test_data) - 1):
    combined_data = pd.concat([current_train_data, test_data.iloc[[i]]])
    y_hat_ret = nf.predict(combined_data)
    y_hat_test_ret = pd.concat([y_hat_test_ret, y_hat_ret.iloc[[-1]]])
    current_train_data = combined_data

predicted_returns = y_hat_test_ret['TFT'].values

predicted_prices_ret = []
for i, ret in enumerate(predicted_returns):
    if i == 0:
        last_true_price = train_data['price'].iloc[-1]
    else:
        last_true_price = test_data['price'].iloc[i-1]
    predicted_prices_ret.append(last_true_price * (1 + ret))

To visualize our results, we create a graph comparing the model’s predictions (red) with the actual S&P 500 prices (green).

In [ ]:
import matplotlib.pyplot as plt
true_values = test_data['price']

plt.figure(figsize=(12, 6))
plt.plot(train_data['ds'], train_data['price'], label='Training Data', color='blue')
plt.plot(test_data['ds'], true_values, label='True Prices', color='green')
plt.plot(test_data['ds'], predicted_prices_ret, label='Predicted Prices', color='red')
plt.legend()
plt.title('Basic SPY Stepwise Forecast using TFT')
plt.xlabel('Date')
plt.ylabel('SPY Price')
plt.savefig('spy_forecast_chart.png', dpi=300, bbox_inches='tight')
plt.close()

## Conclusion
This is it! We’ve successfully learned how to acquire and use free, reliable financial and economic data for our stock forecasting model. Specifically, we covered:

- Retrieving high-quality financial and economic data from the FRED API at no cost
- Processing and cleaning the data for machine learning use
- Implementing dimensionality reduction using PCA
- Training a stock forecasting model with our acquired data

In upcoming articles, we’ll expand our approach by:

- Incorporating additional data sources, such as social media sentiment analysis
- Exploring more sophisticated dimensionality reduction techniques tailored for deep learning models